# =========================================================
# EMAIL THREAT INTELLIGENCE
# INFERENCE PIPELINE
# =========================================================

Goal:
Build a reusable production-ready spam prediction pipeline.

This notebook will:

- load trained artifacts
- preprocess raw emails
- engineer advanced features
- vectorize text
- combine sparse features
- predict spam probability
- generate threat levels

This pipeline will later be moved into:

backend/services/inference.py

In [1]:
# =========================================================
# IMPORT LIBRARIES
# =========================================================

import re
import string
import joblib
import numpy as np
import pandas as pd

from scipy.sparse import hstack

from nltk.corpus import stopwords

from nltk.stem import PorterStemmer

In [2]:
# =========================================================
# LOAD ARTIFACTS
# =========================================================

advanced_vectorizer = joblib.load(

    "../artifacts/"
    "advanced_hybrid_tfidf_vectorizer.pkl"
)

advanced_scaler = joblib.load(

    "../artifacts/"
    "advanced_manual_feature_scaler.pkl"
)

advanced_model = joblib.load(

    "../models/"
    "advanced_xgboost_model.pkl"
)

advanced_label_encoder = joblib.load(

    "../artifacts/"
    "advanced_hybrid_label_encoder.pkl"
)

print(

    "Artifacts loaded successfully."
)

Artifacts loaded successfully.


In [3]:
# =========================================================
# NLP OBJECTS
# =========================================================

stemmer = PorterStemmer()

stop_words = set(

    stopwords.words("english")
)

In [4]:
# =========================================================
# CLEAN TEXT
# =========================================================

def clean_text(text):

    if not isinstance(text, str):

        return ""

    text = text.lower()

    text = re.sub(

        r"http\S+|www\S+",

        " ",

        text
    )

    text = re.sub(

        r"\S+@\S+",

        " ",

        text
    )

    text = re.sub(

        r"<.*?>",

        " ",

        text
    )

    text = re.sub(

        r"\d+",

        " ",

        text
    )

    text = text.translate(

        str.maketrans(

            "",

            "",

            string.punctuation
        )
    )

    text = re.sub(

        r"\s+",

        " ",

        text
    )

    tokens = text.split()

    cleaned_tokens = []

    for token in tokens:

        if token not in stop_words:

            stemmed_word = stemmer.stem(token)

            cleaned_tokens.append(

                stemmed_word
            )

    return " ".join(cleaned_tokens)

In [5]:
# =========================================================
# URL COUNT
# =========================================================

def count_urls(text):

    urls = re.findall(

        r"http[s]?://\S+|www\.\S+",

        str(text)
    )

    return len(urls)


# =========================================================
# EXCLAMATION COUNT
# =========================================================

def count_exclamations(text):

    return str(text).count("!")


# =========================================================
# UPPERCASE RATIO
# =========================================================

def uppercase_ratio(text):

    text = str(text)

    if len(text) == 0:

        return 0

    upper_count = sum(

        1

        for char in text

        if char.isupper()
    )

    return upper_count / len(text)


# =========================================================
# HTML TAG COUNT
# =========================================================

def count_html_tags(text):

    tags = re.findall(

        r"<[^>]+>",

        str(text)
    )

    return len(tags)


# =========================================================
# SPECIAL CHARACTER COUNT
# =========================================================

def count_special_chars(text):

    special_chars = re.findall(

        r"[^a-zA-Z0-9\s]",

        str(text)
    )

    return len(special_chars)


# =========================================================
# SPAM KEYWORD COUNT
# =========================================================

spam_keywords = [

    "free",
    "win",
    "winner",
    "money",
    "offer",
    "urgent",
    "click",
    "buy",
    "cash",
    "prize"
]

def count_spam_keywords(text):

    text = str(text).lower()

    count = 0

    for word in spam_keywords:

        count += text.count(word)

    return count

In [6]:
# =========================================================
# THREAT LEVEL FUNCTION
# =========================================================

def get_threat_level(probability):

    if probability >= 0.90:

        return "HIGH"

    elif probability >= 0.60:

        return "MEDIUM"

    else:

        return "LOW"

In [10]:
# =========================================================
# MAIN PREDICTION PIPELINE
# =========================================================

def predict_email(email_text):

    # =====================================================
    # CLEAN TEXT
    # =====================================================

    cleaned_text = clean_text(

        email_text
    )

    # =====================================================
    # TF-IDF TRANSFORM
    # =====================================================

    text_vector = advanced_vectorizer.transform(

        [cleaned_text]
    )

    # =====================================================
    # CREATE MANUAL FEATURES
    # =====================================================

    manual_features = pd.DataFrame([{

        "url_count": count_urls(email_text),

        "exclamation_count": count_exclamations(email_text),

        "uppercase_ratio": uppercase_ratio(email_text),

        "html_tag_count": count_html_tags(email_text),

        "special_char_count": count_special_chars(email_text),

        "spam_keyword_count": count_spam_keywords(email_text)
    }])

    # =====================================================
    # SCALE FEATURES
    # =====================================================

    scaled_manual_features = (

        advanced_scaler.transform(

            manual_features
        )
    )

    # =====================================================
    # COMBINE FEATURES
    # =====================================================

    combined_features = hstack([

        text_vector,

        scaled_manual_features

    ]).tocsr()

    # =====================================================
    # PREDICT PROBABILITY
    # =====================================================

    probability = (

        advanced_model
        .predict_proba(combined_features)[0][1]
    )

    # =====================================================
    # CUSTOM THRESHOLD
    # =====================================================

    if probability >= 0.70:

        prediction = 1

    else:

        prediction = 0

    # =====================================================
    # DECODE LABEL
    # =====================================================

    label = (

        advanced_label_encoder
        .inverse_transform([prediction])[0]
    )

    # =====================================================
    # THREAT LEVEL
    # =====================================================

    threat_level = get_threat_level(

        probability
    )

    # =====================================================
    # RETURN RESULTS
    # =====================================================

    return {

        "prediction": label,

        "spam_probability": round(

            float(probability),
            4
        ),

        "threat_level": threat_level,

        "features": {

            "url_count": int(

                manual_features["url_count"][0]
            ),

            "exclamation_count": int(

                manual_features["exclamation_count"][0]
            ),

            "uppercase_ratio": round(

                float(
                    manual_features[
                        "uppercase_ratio"
                    ][0]
                ),
                4
            ),

            "html_tag_count": int(

                manual_features[
                    "html_tag_count"
                ][0]
            ),

            "special_char_count": int(

                manual_features[
                    "special_char_count"
                ][0]
            ),

            "spam_keyword_count": int(

                manual_features[
                    "spam_keyword_count"
                ][0]
            )
        }
    }

In [11]:
# =========================================================
# TEST EMAIL
# =========================================================

sample_email = """

CONGRATULATIONS!!!

You won a FREE iPhone.

Click here immediately:
http://spam-offer.com

Claim your prize NOW!!!

"""

result = predict_email(

    sample_email
)

result

{'prediction': 'spam',
 'spam_probability': 0.997,
 'threat_level': 'HIGH',
 'features': {'url_count': 1,
  'exclamation_count': 6,
  'uppercase_ratio': 0.2203,
  'html_tag_count': 0,
  'special_char_count': 13,
  'spam_keyword_count': 4}}

In [12]:
# =========================================================
# HAM EMAIL TEST
# =========================================================

ham_email = """

Hi team,

Please find the project update attached.

Let me know if any changes are required.

Thanks,
Prasanna

"""

result = predict_email(

    ham_email
)

result

{'prediction': 'ham',
 'spam_probability': 0.536,
 'threat_level': 'LOW',
 'features': {'url_count': 0,
  'exclamation_count': 0,
  'uppercase_ratio': 0.0439,
  'html_tag_count': 0,
  'special_char_count': 4,
  'spam_keyword_count': 0}}